# Week 6 — Capstone: Academic Assistant Agent

> **Source notebook for** [`src/academic_assistant/`](../src/academic_assistant/).

This is the integration week. We compose every component built in Weeks 1–5 into a single autonomous system that, given a research topic, produces a structured Markdown literature review.

## The pipeline

```
        topic
          │
          ▼
  ┌────────────────┐   arXiv API
  │ arxiv_ingest   │◀───────────────  (Week 6)
  └───────┬────────┘
          │ Paper objects
          ▼
  ┌────────────────┐
  │  PaperStore    │  SQLite, exposed read-only via MCP  (Week 1)
  └───────┬────────┘
          │
          ▼
  ┌────────────────┐
  │ HierarchicalRAG│  parent-child chunks + Chroma       (Week 2)
  └───────┬────────┘
          │ candidates
          ▼
  ┌────────────────┐
  │ CrossEncoder   │  precision re-ranking               (Week 3)
  │  Reranker      │
  └───────┬────────┘
          │ top-k per section
          ▼
  ┌────────────────┐
  │  LLM synthesis │  ReAct-style reasoning              (Weeks 4–5)
  └───────┬────────┘
          │
          ▼
   Markdown review
```


## Learning objectives

1. Compose independently-tested modules into an end-to-end autonomous system.
2. Use the MCP SQLite resource layer to persist and expose ingested papers.
3. Drive an outline-then-synthesize generation strategy that constrains the LLM to grounded claims.
4. Reason about the failure modes of the *composed* system, which differ from those of its parts.


## 1. Ingestion: arXiv → structured Papers

[`src/academic_assistant/arxiv_ingest.py`](../src/academic_assistant/arxiv_ingest.py) wraps the arXiv API and yields typed `Paper` objects. In the notebook we use a small fixture instead of a live call so the notebook is reproducible offline; in production you'd call `fetch_papers(topic, max_results=25)`.


In [ ]:
from src.academic_assistant.arxiv_ingest import Paper

# A small offline fixture standing in for a live arXiv pull.
fixture = [
    Paper("2212.10496", "Precise Zero-Shot Dense Retrieval without Relevance Labels",
          ["Luyu Gao", "Xueguang Ma", "Jimmy Lin", "Jamie Callan"],
          "We propose HyDE, which generates a hypothetical document via an instruction-following "
          "language model and embeds it for retrieval, outperforming dense retrievers without labels.",
          "2022-12-20", "https://arxiv.org/pdf/2212.10496", ["cs.IR"]),
    Paper("2210.03629", "ReAct: Synergizing Reasoning and Acting in Language Models",
          ["Shunyu Yao", "Jeffrey Zhao", "Dian Yu", "Nan Du"],
          "ReAct interleaves reasoning traces and task-specific actions, allowing the model to "
          "induce, track, and update action plans while interfacing with external tools.",
          "2022-10-06", "https://arxiv.org/pdf/2210.03629", ["cs.CL"]),
    Paper("1908.10084", "Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks",
          ["Nils Reimers", "Iryna Gurevych"],
          "Sentence-BERT modifies BERT with siamese and triplet networks to derive semantically "
          "meaningful sentence embeddings comparable with cosine similarity.",
          "2019-08-27", "https://arxiv.org/pdf/1908.10084", ["cs.CL"]),
]
for p in fixture:
    print(f"{p.arxiv_id}: {p.title[:60]}")


## 2. Persistence: the MCP-backed SQLite store

`PaperStore` writes papers to SQLite. The same database is exposed *read-only* to any MCP client through `SQLiteResources` (Week 1) — so a Claude Desktop session connected to our MCP server could browse the ingested papers as a resource.


In [ ]:
import tempfile
from pathlib import Path
from src.academic_assistant.store import PaperStore

db_path = Path(tempfile.mktemp(suffix=".sqlite"))
store = PaperStore(db_path)
store.upsert_many(fixture)

print("rows in store:", len(store.all_papers()))
for row in store.all_papers():
    print(" ", row["arxiv_id"], "-", row["title"][:50])


In [ ]:
# Expose it read-only through the MCP resource layer (Week 1).
from src.mcp_core.resources import SQLiteResources
store.close()   # close the writer before opening a read-only handle

mcp_resources = SQLiteResources(db_path)
print("MCP resources advertised:")
for r in mcp_resources.list_resources():
    print(" ", r["uri"], f"({r['mimeType']})")
print("\nRead via MCP:")
print(mcp_resources.read("sqlite://papers")[0]["title"])


## 3. Indexing and retrieval (Weeks 2–3)

Abstracts are chunked hierarchically, embedded, and stored in Chroma. We then retrieve and re-rank for a sample section query.


In [ ]:
from src.rag_engine.chunking import HierarchicalChunker
from src.rag_engine.embeddings import SentenceEncoder
from src.rag_engine.pipeline import HierarchicalRAG
from src.rag_engine.vector_stores import build_store
from src.reranking.cross_encoder import CrossEncoderReranker
from src.reranking.pipeline import RerankingPipeline

persist = tempfile.mkdtemp()
rag = HierarchicalRAG(
    chunker=HierarchicalChunker(parent_chars=600, child_chars=250),
    encoder=SentenceEncoder("BAAI/bge-small-en-v1.5"),
    child_store=build_store("chroma", collection="capstone", persist_dir=persist),
    parent_lookup={},
)
for p in fixture:
    rag.ingest_document(
        f"# {p.title}\n\n{p.abstract}",
        metadata={"arxiv_id": p.arxiv_id, "title": p.title, "published": p.published},
    )

reranker = RerankingPipeline(reranker=CrossEncoderReranker(), candidate_k=5)

section = "approaches to zero-shot dense retrieval"
candidates = rag.retrieve(section, k=3)
top = reranker.rerank(section, candidates, k=2)
print(f"Section: {section}\n")
for d in top:
    print(f"  [{d.cross_encoder_score:+.3f}] {d.metadata['title'][:60]}")


## 4. The outline-then-synthesize strategy

A naive "summarize all these papers" prompt produces a shapeless wall of text. Instead the assistant:

1. Asks the LLM for a **section outline** (4–6 titles that tell a coherent story).
2. For each section, retrieves + re-ranks the most relevant excerpts.
3. Asks the LLM to write *that one section*, grounded **only** in the retrieved excerpts.
4. Appends a References section built from the structured `Paper` metadata.

This structure is what keeps the output grounded: by feeding the synthesis step a small set of re-ranked excerpts and forbidding fabricated citations, we constrain the LLM to claims it can support. The full implementation is in [`src/academic_assistant/pipeline.py`](../src/academic_assistant/pipeline.py).


In [ ]:
from src.academic_assistant.pipeline import AcademicAssistant, AssistantConfig

# Scripted LLM so the notebook runs offline. Production uses build_client(...).
class FakeLLM:
    def __init__(self, scripted): self.scripted = list(scripted)
    def complete(self, messages, **kw):
        return self.scripted.pop(0) if self.scripted else "(no more scripted responses)"

outline = "Dense Retrieval Foundations\nZero-Shot Retrieval with HyDE\nReasoning and Acting Agents"
section_text = (
    "## Section\n\nRecent work establishes sentence embeddings as the backbone of dense "
    "retrieval [Reimers & Gurevych, 2019]. Building on this, HyDE removes the need for "
    "relevance labels by embedding a generated hypothetical document [Gao et al., 2022]."
)
fake = FakeLLM([outline] + [section_text] * 3)

assistant = AcademicAssistant(llm=fake)
# We call the internal methods directly to avoid a live arXiv fetch in the notebook.
sections = assistant._build_outline("agentic retrieval", 3)
print("OUTLINE:")
for s in sections:
    print("  -", s)


In [ ]:
# Synthesize one section from the re-ranked excerpts.
md_section = assistant._synthesize_section(sections[1], top)
print(md_section)


In [ ]:
# The references section is built deterministically from structured metadata —
# no LLM involved, so citations cannot be hallucinated.
print(assistant._references_section(fixture))


## 5. Running the whole thing

In production the entire pipeline runs from the command line:

```bash
python -m src.academic_assistant.cli \
    --topic "self-rewarding language models" \
    --max-papers 25 \
    --sections 5 \
    --output reports/self_rewarding_lm_review.md
```

This single command: pulls 25 papers from arXiv, persists them to the MCP-exposed SQLite store, indexes their abstracts with hierarchical chunking, builds a 5-section outline, retrieves and cross-encoder-reranks excerpts per section, synthesizes each section with grounded citations, and writes a complete Markdown literature review.


## 6. Failure modes of the *composed* system

Composition introduces failure modes that none of the parts exhibit alone:

| Failure | Where it emerges | Mitigation |
|---------|------------------|-----------|
| Empty retrieval for a section | Outline proposes a section the corpus doesn't cover | Skip-or-flag sections with no high-scoring candidates |
| Citation drift | LLM cites a paper not in the excerpts | References built from structured metadata, not LLM output |
| Topic collapse | All sections retrieve the same 2 papers | Diversify with MMR (maximal marginal relevance) at retrieval |
| Cost blow-up | sections × candidate_k LLM calls | Cache embeddings; batch the cross-encoder; bound section count |
| Stale index | Re-running on a new topic reuses an old collection | Collection name keyed on topic hash (see `pipeline.py`) |

The lesson: **integration testing is not the sum of unit tests.** Each module is correct in isolation; the *composition* needs its own tests and its own guards.


## 7. Exercises

1. **MMR diversification.** Add maximal marginal relevance to `HierarchicalRAG.retrieve` so sections don't all pull the same papers. Tune the diversity/relevance $\lambda$.
2. **Agentic synthesis.** Replace the single-shot section synthesis with the Week 4 ReAct agent, giving it a `search_corpus` tool so it can pull *additional* evidence mid-section.
3. **Self-critique.** Add the Week 5 Critic agent as a final pass over the generated review, checking each citation against the excerpts and flagging unsupported claims.
4. **PDF ingestion.** Extend `arxiv_ingest` to download and parse full PDFs (not just abstracts) using the `pdf` skill, then re-run. How much does full-text retrieval improve section quality over abstract-only?


## 8. Course retrospective

Over six weeks we built, from first principles:

1. **MCP** — a JSON-RPC protocol server exposing tools and resources to an LLM.
2. **Hierarchical RAG** — chunking strategies, embedding geometry, and a from-scratch HNSW index.
3. **Re-ranking** — the Bi/Cross-Encoder duality, query transformation, and HyDE.
4. **ReAct** — a pure-Python reasoning-and-acting loop with robust failure handling.
5. **Multi-agent systems** — a self-correcting Coder–Executor–Critic loop with provable termination.
6. **The capstone** — all of the above, composed into an autonomous research assistant.

No high-level agent framework was used anywhere. Every abstraction in this repository is one you wrote and own. That is the difference between *calling* an agent library and *understanding* one — and it is what the "from scratch" in the repository name is meant to convey.

➡ The natural next step is to deploy the capstone behind the MCP server from Week 1, so any MCP-compliant client can invoke your research assistant as a tool.
